# Exercises on Transformers and NLP

## Exercise 1: Pretrained Word Embeddings
This is a quick exercise where you can play around with pretrained word embeddings such as GloVe or Word2Vec. The goal is to understand how word embeddings capture semantic relationships between words.

In [ ]:
import gensim.downloader
w2v = gensim.downloader.load('glove-wiki-gigaword-50')
# w2v = gensim.downloader.load('word2vec-google-news-300')

res = w2v['king'] - w2v['man'] + w2v['woman']

w2v.most_similar(positive=res, topn=5)

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b)/(np.linalg.norm(a) * np.linalg.norm(b))
    
cosine_similarity(w2v['nurse'], w2v['physicist'])

## Exercise 2: Contextualized word embeddings
In this task you will implement word embeddings using PyTorch's `nn.Embedding` module. We provide most of the code (the model is rather trivial, and the training loop fairly standard anyways), your task is be to prepare the training data in the appropriate form, and intrepret the final results.

We use a very simple set of sentences as our text corpus, essentially each sentence is a semantic tuple; or rather nodes connected by concepts in a heterogeneous graph.
Once you are done with the first pass of the exercise, you can come back and add more connections to the text corpus to see how it affects the learned embeddings, or try to make the sentences more complex (e.g., longer sentences, more context words, etc.).

In [ ]:
simple_text_data = """
bananas are fruit
apples are fruit
grapes are fruit
oranges are fruit
lemons are fruit
bob likes fruit
bananas are yellow
apples are red
grapes are purple
oranges are orange
lemons are yellow
alice likes apples
carrol hates lemons
dave likes grapes
dave likes lemons
bob is a boy
alice is a girl
carrol is a girl
dave is a boy
"""

In [ ]:
# Sentences, vocabulary and training data preparation
sentences = [line.strip().split() for line in simple_text_data.strip().split('\n')]
vocab = set(word for sentence in sentences for word in sentence)

word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}

def generate_semantic_tuples(sentence: str, word2idx: dict, context_size: int) -> list:
    """ Your code should take a sentence (list of words), a word-to-index mapping (dict), and a context size (int) as input,
        and return a list of tuples (context_words, target_word), where context_words are words 
        in the sentence near the target_word within the given context_size.

        Example input: 
            sentence = ["bananas", "are", "fruit"]
            context_size = 2
        Abstracted output:
            [
                (["are", "fruit"], "bananas"),
                (["bananas", "fruit"], "are"),
                (["bananas", "are"], "fruit")
            ]
        Actual output will use word indices instead of words.
        Example actual output:
            [
                ([8, 5], 3), ([3, 5], 8), ([3, 8], 5)
            ]
    """
    ### YOUR CODE HERE ###




data = []
for sentence in sentences:
    tuples = generate_semantic_tuples(sentence, word2idx=word_to_ix, context_size=2)
    data.extend(tuples)

print(data)

In [ ]:
# DataLoader using the prepared data
import torch
from torch.utils.data import DataLoader, Dataset

class WordDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)
    
dataset = WordDataset(data)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True)

In [ ]:
# Training loop
import torch
import torch.nn as nn

# This here is a simple word embedding model
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).mean(dim=1)
        out = self.linear(embeds)
        return out

# Training parameters
num_epochs = 100
embedding_dim = 2
vocab_size = len(vocab)

model = CBOW(vocab_size, embedding_dim)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(num_epochs):
    total_loss = 0
    for context, target in data_loader:
        model.zero_grad()
        log_probs = model(context)
        loss = loss_function(log_probs, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss:.4f}')

In [ ]:
# Visualization of the embeddings
embeddings = model.embeddings.weight.data.numpy()

import matplotlib.pyplot as plt
for i, word in enumerate(vocab):
    plt.scatter(embeddings[i, 0], embeddings[i, 1])
    plt.annotate(word, (embeddings[i, 0], embeddings[i, 1]))
    
plt.xlabel("Embedding Dimension 1")
plt.ylabel("Embedding Dimension 2")
plt.show()

## Exercise 3: Long-Short Term Memory (LSTM) Networks
This exercise focuses on building and training a simple RNN model for sequence prediction tasks. This task follows the excellent tutorial found here: [Machine Learning Mastery](https://machinelearningmastery.com/text-generation-with-lstm-in-pytorch/), and is essentially just reading and following along with the tutorial. If you want you can use your own text corpus for training the model, note that larger volumes yield better results, but also require more training time.

In [ ]:
# Import the vocabulary and prepare the data
import numpy as np
 
# load ascii text and covert to lowercase
filename = 
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower()

# If your text is very large, you can use a smaller subset for faster training
# raw_text = raw_text[:100000]  
 
# create mapping of unique chars to integers
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))
 
# summarize the loaded data
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

## Exercise 4: Fine-tuning DistilBERT
In this exercise you will fine-tune a pre-trained DistilBERT model on a small dataset, to see how transfer learning with transformers can be used for text classification tasks.

We are loosely following along with the tutorial found here: https://huggingface.co/blog/sentiment-analysis-python

Note that the code has been modified due to updates in the Huggingface library since the blog post was written.

In [ ]:
# Libraries you need:
# !pip install datasets transformers evaluate

In [ ]:
# Load IMDB dataset
from datasets import load_dataset
imdb = load_dataset("imdb")

# Take a subset
small_train_dataset = imdb["train"].shuffle(seed=42).select([i for i in list(range(1000))])
small_test_dataset = imdb["test"].shuffle(seed=42).select([i for i in list(range(100))])

In [ ]:
# Define wrapper for metrics to be computed during training
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    load_accuracy = evaluate.load("accuracy")
    load_f1 = evaluate.load("f1")
    
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": accuracy, "f1": f1}

In [ ]:
# Get the model and tokenizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Prepare data
def preprocess_function(examples):
   return tokenizer(examples["text"], truncation=True)
 
tokenized_train = small_train_dataset.map(preprocess_function, batched=True)
tokenized_test = small_test_dataset.map(preprocess_function, batched=True)

# Set format for PyTorch
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set up training arguments and trainer
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",             # output directory
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# First check how well the untrained model performs
results = trainer.evaluate()
results

In [ ]:
# Train the model (WARNING: This may take a while depending on your hardware)
trainer.train()

In [ ]:
# Evaluate accuracy on the test set
results = trainer.evaluate()
results

In [ ]:
# Apply the trained model to some example texts
import torch

texts = [
    "I absolutely loved this movie! The plot was thrilling and the characters were well-developed.",
    "This was a terrible film. I wasted two hours of my life watching it.",
    "An average movie with some good moments, but overall it was just okay."
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)
predictions = torch.argmax(outputs.logits, dim=-1)
for text, pred in zip(texts, predictions):
    sentiment = "Positive" if pred.item() == 1 else "Negative"
    print(f'Text: "{text}" => Sentiment: {sentiment}')

## Exercise 5: Transformers in PyTorch

In this exercise you will get to experiment with transformer components in the PyTorch lingo. We will end up with some simple transformer models, that you can in principle use for sequence to sequence tasks, or language modeling. However the main goal is to understand how the transformer architecture works, and how to implement it in PyTorch.

This article on [datacamp](https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch) provides a comprehensive guide to building a transformer model using PyTorch. Feel free to have a look at it for inspiration and code snippets, here we will just steal their PositionalEncoding class.

In [ ]:
# Copy the PositionalEncoding class from the datacamp article here



In [ ]:
# Plot the positional encodings to visualize them
import matplotlib.pyplot as plt

### YOUR CODE HERE ###


def plot_positional_encoding(pos_encoding):
    plt.figure(figsize=(10, 6))
    plt.pcolormesh(pos_encoding.numpy(), cmap='viridis')
    plt.colorbar()
    plt.xlabel('Embedding Dimension')
    plt.ylabel('Position')
    plt.title('Positional Encoding')
    plt.show()

Next you will implement a simple single head attention layer in pytorch, you do not need to make it super robust, just make it work for the simple cases we will test it on.

It may help to have a look at the theoretical exercise first to understand how the attention mechanism works.

In [ ]:
# Dummy Attention Layer Implementation
class AttentionLayer(nn.Module):
    def __init__(self, input_dim, attention_dim):
        super(AttentionLayer, self).__init__()
        self.query = nn.Linear(input_dim, attention_dim)
        self.key = nn.Linear(input_dim, attention_dim)

        self.value = nn.Linear(input_dim, attention_dim)
        self.value_proj = nn.Linear(attention_dim, input_dim)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """ Implement the forward pass of the attention layer."""
        ### YOUR CODE HERE ###
        
        
        return output, attn_weights
    
# Check that it is working as intended
batch_size = 2
seq_len = 5
input_dim = 10
attention_dim = 8

x = torch.randn(batch_size, seq_len, input_dim)

attention_layer = AttentionLayer(input_dim, attention_dim)
output, attn_weights = attention_layer(x)

print("Output shape:", output.shape)          # Expected: (batch_size, seq_len, input_dim)
print("Attention weights shape:", attn_weights.shape)  # Expected: (batch_size, seq_len, seq_len)

Next is a minimal encoder-decoder transformer model implementation in Pytorch. Look at the documentation for `nn.TransformerEncoderLayer` and `nn.TransformerDecoderLayer` to understand what these components hide under the hood. Explain the approach used to the positional encoding in this example.

In [ ]:
# Minimal encoder-decoder transformer model in PyTorch
import torch
import torch.nn as nn

class Transformer(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, max_seq_length):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(input_dim, model_dim)
        self.decoder_embedding = nn.Embedding(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_length, model_dim))

        # set batch_first=True so layers accept (batch, seq, model_dim)
        self.encoder_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(model_dim, num_heads, batch_first=True) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(model_dim, num_heads, batch_first=True) for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(model_dim, input_dim)

    def forward(self, src, tgt):
        # src, tgt are expected as LongTensors with shape (batch, seq_len)
        src = self.encoder_embedding(src) + self.positional_encoding[:, :src.size(1), :]
        tgt = self.decoder_embedding(tgt) + self.positional_encoding[:, :tgt.size(1), :]
        for layer in self.encoder_layers:
            src = layer(src)
        for layer in self.decoder_layers:
            tgt = layer(tgt, src)
        return self.fc_out(tgt)


In the final part, we provide you with a simple Decoder-only transformer model implementation in PyTorch. 

If you want, you can train this model on some simple text data and evaluate its performance. Use any text corpus you like, but keep in mind that larger corpora will yield better results, but also require more training time. Moreover, there are several hyper-parameters you can experiment with, such as the number of layers, number of attention heads, embedding dimensions, etc. Try to see how these parameters affect the model's performance.

In [ ]:
# Minimal decoder-only transformer model in PyTorch
import torch
import torch.nn as nn

class TransformerDecoderOnly(nn.Module):
    def __init__(self, input_dim, model_dim, num_heads, num_layers, max_seq_length=1000):
        super(TransformerDecoderOnly, self).__init__()
        self.embedding = nn.Embedding(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_length, model_dim))

        # set batch_first=True so layers accept (batch, seq, model_dim)
        self.decoder_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(model_dim, num_heads, batch_first=True) for _ in range(num_layers)
        ])
        self.fc_out = nn.Linear(model_dim, input_dim)

    def forward(self, tgt):
        # tgt is expected as LongTensor with shape (batch, seq_len)
        tgt = self.embedding(tgt) + self.positional_encoding[:, :tgt.size(1), :]
        for layer in self.decoder_layers:
            tgt = layer(tgt, tgt)
        return self.fc_out(tgt)


In [ ]:
# Big block of code I stole from somewhere that tokenizes text data and prepares it for training

# TODO: set path to your text file here
raw_text_path = 

with open(raw_text_path, 'r', encoding='utf-8') as file:
    text = file.read().lower()

# --- Simple regex tokenizer (keeps words and punctuation) ---
import re
token_pattern = re.compile(r"\w+|[^\w\s]", re.UNICODE)
tokens = token_pattern.findall(text)

# --- Vocabulary with special tokens ---
PAD = "<pad>"
BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"

vocab_tokens = [PAD, BOS, EOS, UNK] + sorted(set(tokens))
token_to_id = {t: i for i, t in enumerate(vocab_tokens)}
id_to_token = {i: t for t, i in token_to_id.items()}
vocab_size = len(token_to_id)
print(f"Vocab size: {vocab_size}")

# --- Encode helper ---
def encode(tok_list):
    return [token_to_id.get(t, token_to_id[UNK]) for t in tok_list]

# --- Prepare sliding-window sequence pairs for causal language modeling ---
# TODO: these are hyperparameters you can tune
max_len = 64         # sequence length (tune for memory)
stride = 32          # overlap between windows

encoded = encode(tokens)

examples = []
for i in range(0, max(1, len(encoded) - 1), stride):
    window = encoded[i:i+max_len]
    if len(window) == 0:
        break
    # For causal language modeling:
    input_seq = [token_to_id[BOS]] + window[:-1]  # input sequence with BOS
    target_seq = window                           # target sequence
    examples.append((input_seq, target_seq))

print(f"Prepared {len(examples)} examples (max_len={max_len}, stride={stride})")

# --- PyTorch Dataset + collate (padding + masks) ---
import torch
from torch.utils.data import Dataset, DataLoader

pad_idx = token_to_id[PAD]

class CausalLMDataset(Dataset):
    def __init__(self, exs):
        self.exs = exs
    def __len__(self):
        return len(self.exs)
    def __getitem__(self, i):
        return self.exs[i]

def collate_batch(batch):
    # batch is list of (input_seq, target_seq)
    input_seqs, target_seqs = zip(*batch)
    input_lens = [len(seq) for seq in input_seqs]
    max_len = max(input_lens)

    input_padded = torch.full((len(batch), max_len), pad_idx, dtype=torch.long)
    target_padded = torch.full((len(batch), max_len), pad_idx, dtype=torch.long)

    for i, (inp, tgt) in enumerate(zip(input_seqs, target_seqs)):
        input_padded[i, :len(inp)] = torch.tensor(inp, dtype=torch.long)
        target_padded[i, :len(tgt)] = torch.tensor(tgt, dtype=torch.long)

    # padding mask (True where padding) -> shape (batch, seq_len)
    attention_mask = (input_padded != pad_idx)

    # subsequent (causal) mask for decoder: shape (seq_len, seq_len), True where to mask
    def subsequent_mask(size):
        return torch.triu(torch.ones(size, size, dtype=torch.bool), diagonal=1)

    causal_mask = subsequent_mask(max_len)  # will be broadcasted internally

    return {
        "input_ids": input_padded,          # shape (batch, seq_len)
        "labels": target_padded,           # shape (batch, seq_len)
        "attention_mask": attention_mask,  # shape (batch, seq_len)
        "causal_mask": causal_mask         # shape (seq_len, seq_len)
    }

dataset = CausalLMDataset(examples)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_batch)

# --- Example: get one batch and show shapes ---
batch = next(iter(dataloader))
print({k: (v.shape if isinstance(v, torch.Tensor) else type(v)) for k, v in batch.items()})

In [ ]:
# Training loop (should be reminicent of previous ones)
# TODO: main hyperparameters
input_dim = vocab_size
model_dim = 256
num_heads = 4
num_layers = 3
max_seq_length = max_len

model = TransformerDecoderOnly(input_dim, model_dim, num_heads, num_layers, max_seq_length)
loss_function = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

# training loop
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"]
        labels = batch["labels"]
        attention_mask = batch["attention_mask"]
        causal_mask = batch["causal_mask"]

        optimizer.zero_grad()
        output = model(input_ids)
        output = output.view(-1, input_dim)
        labels = labels.view(-1)
        loss = loss_function(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")


In [ ]:
# Example of text generation
model.eval()

# TODO: adjust the starting prompt here
num_tokens_to_generate = 50
prompt = "Once upon a time "

print("Prompt: ", prompt)
input_ids = torch.tensor([token_to_id.get(t, token_to_id[UNK]) for t in prompt.split()], dtype=torch.long).unsqueeze(0)

with torch.no_grad():
    generated = input_ids
    for _ in range(num_tokens_to_generate):  # generate 50 tokens
        tgt_in = generated
        tgt_key_padding_mask = (tgt_in == pad_idx)
        tgt_mask = torch.triu(torch.ones(tgt_in.size(1), tgt_in.size(1), dtype=torch.bool), diagonal=1)

        output = model(tgt_in)
        next_token_logits = output[0, -1, :]
        next_token_id = next_token_logits.argmax().unsqueeze(0).unsqueeze(0)
        generated = torch.cat((generated, next_token_id), dim=1)

    generated_tokens = [id_to_token[idx.item()] for idx in generated[0]]
    print("Generated text: ", " ".join(generated_tokens))